In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.1 RoPE

Learned position embeddings are a lookup table capped at `block_size`. **Rotary**
position embeddings (RoPE) instead *rotate* the query and key vectors by an angle
proportional to position. Attention scores then depend on relative position, and
the model can extrapolate past its training length.

In [ ]:
from pocketlm import precompute_rope, apply_rope, PocketLM, PocketLMConfig

cos, sin = precompute_rope(head_size=8, seq_len=16)
x = torch.randn(1, 2, 5, 8)   # [batch, heads, tokens, head_size]
rotated = apply_rope(x, cos, sin)
print("norm preserved (rotation is length-preserving):",
      torch.allclose(rotated.norm(dim=-1), x.norm(dim=-1), atol=1e-5))
model = PocketLM(PocketLMConfig(vocab_size=20, block_size=16, n_layer=2,
                                n_head=2, n_embd=32, pos="rope"))
print("rope model has a learned pos table:", hasattr(model, "pos_emb"))

RoPE adds no parameters and injects position *inside* attention, so a rope model
drops the learned position table entirely.

In [ ]:
assert torch.allclose(rotated.norm(dim=-1), x.norm(dim=-1), atol=1e-5)
assert not hasattr(model, "pos_emb")